# RT-DETR on DocLayNet

Settings: GPU T4 x2, Internet on.

Run with Save Version -> Save & Run All (Commit), not interactive -
survives the tab closing.

## Check hardware

Kaggle doesn't always give the GPU you picked, so confirming this
matters for the reproducibility claim.

In [ ]:
!nvidia-smi

## Install

Pinning ultralytics - RT-DETR's args shift between versions.

In [ ]:
!pip install -q ultralytics==8.3.40 "datasets<4.0.0"


## Pull the repo

Calls the repo's scripts directly so the notebook can't drift from
how the data was actually prepared.

In [ ]:
REPO_URL = "https://github.com/RISHIVELS/doc_layout_detection.git"

# %cd into a directory this cell is about to rm -rf breaks the shell's
# own working directory on the *next* run of this cell (getcwd fails
# because the path it thinks it is standing in no longer exists). So I
# step out to a stable parent directory before deleting anything.
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1


## Build the dataset

~3.8GB first run. `--verify 12` renders sample labels to eyeball next.

In [ ]:
!python scripts/prepare_dataset.py --out /kaggle/working/data/doclaynet --verify 12

## Stop and check the labels

My class order is inferred, not confirmed. If it's off by one,
`Table` silently becomes `Section-header` everywhere and training
won't complain. Just look: is the `Table` box actually on a table?

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

checks = sorted(Path("/kaggle/working/data/doclaynet/label_check").glob("*.png"))
print(f"{len(checks)} pages to check\n")

for path in checks[:4]:
    display(Image.open(path).resize((620, 620)))

## Train

RT-DETR-L, 30 epochs, 640px, AMP. 640px loses detail on thin classes
like Footnote - a real tradeoff for the time budget, not hidden.

Checkpoints every epoch so a dropped session loses one epoch, not
the whole run.

In [ ]:
!python scripts/train.py \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --epochs 30 \
    --batch 8 \
    --imgsz 640 \
    --device 0 \
    --name rtdetr_doclaynet

## Evaluate

Per-class AP, per-category mAP, query saturation. Category breakdown
matters most - aggregate mAP can't tell you if it learned structure
or just financial reports.

In [ ]:
!python scripts/evaluate.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --out reports

## Mine failure cases

Ranks worst predictions, renders them against ground truth. Memo's
failure cases come from here, not guessing.

In [ ]:
!python scripts/mine_failures.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet \
    --out reports/failures \
    --top 25

## Collect outputs

Weights, metrics, run metadata, failure renders.

In [ ]:
import shutil
from pathlib import Path

run = Path("runs/detect/rtdetr_doclaynet")
out = Path("/kaggle/working/submission")
out.mkdir(exist_ok=True)

shutil.copy(run / "weights/best.pt", out / "best.pt")
shutil.copy(run / "run_metadata.json", out / "run_metadata.json")
if Path("reports").exists():
    shutil.copytree("reports", out / "reports", dirs_exist_ok=True)

for path in sorted(out.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.1f} MB  {path.relative_to(out)}")